# Demo 07 : Testing agents

Run the cells in order and watch. There is nothing to write.

## a · scripted-fake-model

We script tool calls with `GenericFakeChatModel`. Its `bind_tools` method needs a small subclass for agents with tools. These tests exercise the tool and graph wiring, without testing the real model's choices.

In [1]:
import shutil
import urllib.request

from pathlib import Path
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "data/demo-world/service.db").exists()), None)
DB = here / "service.db"
if root:
    shutil.copyfile(root / "data/demo-world/service.db", DB)
elif not DB.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/data/demo-world/service.db", DB)

In [2]:
from contextlib import closing
import sqlite3
from pathlib import Path
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolCall
from langchain.tools import tool
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langgraph.checkpoint.memory import InMemorySaver


class FakeToolModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


def stock_tool(db):
    @tool
    def check_stock(part_no: str) -> str:
        """Read recorded units in stock for a Veenhaven spare part."""
        with closing(sqlite3.connect(db)) as conn:
            row = conn.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()
        return str(row[0]) if row else "Unknown part"
    return check_stock


def scripted(db, *messages):
    return create_agent(FakeToolModel(messages=iter(messages)), tools=[stock_tool(db)],
                        checkpointer=InMemorySaver())


def call(part_no="CE-DA-03", call_id="stock-1"):
    return AIMessage("", tool_calls=[ToolCall(name="check_stock", args={"part_no": part_no}, id=call_id)])

In [3]:
try:
    plain = GenericFakeChatModel(messages=iter([AIMessage("Hello.")]))
    create_agent(plain, tools=[stock_tool(DB)]).invoke({"messages": [HumanMessage("Stock?")]})
except NotImplementedError:
    print("plain fake: bind_tools is not implemented")
agent = scripted(DB, call(), AIMessage("CE-DA-03 is out of stock."))
config = {"configurable": {"thread_id": "stock-demo"}}
out = agent.invoke({"messages": [HumanMessage("Stock of CE-DA-03?")]}, config)
for message in out["messages"]:
    print(message.type, message.content or getattr(message, "tool_calls", []))
print("saved messages:", len(agent.get_state(config).values["messages"]))

plain fake: bind_tools is not implemented
human Stock of CE-DA-03?
ai [{'name': 'check_stock', 'args': {'part_no': 'CE-DA-03'}, 'id': 'stock-1', 'type': 'tool_call'}]
tool 0
ai CE-DA-03 is out of stock.
saved messages: 4


## b · pytest

We write a test file next to the notebook and run pytest in a subprocess. We assert the real tool result, tool-call correspondence, saved history and thread isolation. The final answer is scripted and cannot prove answer quality.

In [4]:
from pathlib import Path
import subprocess
import sys

test_source = '\nfrom contextlib import closing\nimport sqlite3\nfrom pathlib import Path\nfrom langchain.agents import create_agent\nfrom langchain.messages import AIMessage, HumanMessage, ToolCall\nfrom langchain.tools import tool\nfrom langchain_core.language_models.fake_chat_models import GenericFakeChatModel\nfrom langgraph.checkpoint.memory import InMemorySaver\n\n\nclass FakeToolModel(GenericFakeChatModel):\n    def bind_tools(self, tools, **kwargs):\n        return self\n\n\ndef stock_tool(db):\n    @tool\n    def check_stock(part_no: str) -> str:\n        """Read recorded units in stock for a Veenhaven spare part."""\n        with closing(sqlite3.connect(db)) as conn:\n            row = conn.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()\n        return str(row[0]) if row else "Unknown part"\n    return check_stock\n\n\ndef scripted(db, *messages):\n    return create_agent(FakeToolModel(messages=iter(messages)), tools=[stock_tool(db)],\n                        checkpointer=InMemorySaver())\n\n\ndef call(part_no="CE-DA-03", call_id="stock-1"):\n    return AIMessage("", tool_calls=[ToolCall(name="check_stock", args={"part_no": part_no}, id=call_id)])\n\nimport pytest\n\n\n@pytest.fixture\ndef db(tmp_path):\n    import shutil\n    import urllib.request\n\n    here = Path(__file__).resolve().parent\n    root = next((p for p in [here, *here.parents] if (p / "data/demo-world/service.db").exists()), None)\n    target = tmp_path / "service.db"\n    if root:\n        shutil.copyfile(root / "data/demo-world/service.db", target)\n    else:\n        urllib.request.urlretrieve("https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/data/demo-world/service.db", target)\n    return target\n\n\ndef test_tool_result_and_call_id(db):\n    agent = scripted(db, call(), AIMessage("CE-DA-03 has no stock."))\n    out = agent.invoke({"messages": [HumanMessage("Stock of CE-DA-03?")]},\n                       {"configurable": {"thread_id": "stock"}})\n    results = [m for m in out["messages"] if m.type == "tool"]\n    assert len(results) == 1\n    assert results[0].content == "0"\n    assert results[0].tool_call_id == "stock-1"\n\n\ndef test_unknown_part_is_explicit(db):\n    assert stock_tool(db).invoke({"part_no": "UNKNOWN"}) == "Unknown part"\n\n\ndef test_memory_and_thread_isolation(db):\n    agent = scripted(db, call(), AIMessage("No stock."), AIMessage("CE-DA-03."), AIMessage("No earlier lookup."))\n    config = {"configurable": {"thread_id": "first"}}\n    agent.invoke({"messages": [HumanMessage("Stock?")]}, config)\n    agent.invoke({"messages": [HumanMessage("Which part?")]}, config)\n    saved = agent.get_state(config).values["messages"]\n    assert any(m.type == "tool" and m.content == "0" for m in saved)\n    other = {"configurable": {"thread_id": "other"}}\n    out = agent.invoke({"messages": [HumanMessage("Which part?")]}, other)\n    assert [m.type for m in out["messages"]] == ["human", "ai"]\n    assert len(agent.get_state(config).values["messages"]) == len(saved)\n'
test_path = Path("test_module04_agent.py")
test_path.write_text(test_source, encoding="utf-8")
try:
    run = subprocess.run([sys.executable, "-B", "-m", "pytest", "-p", "no:cacheprovider", "-q", str(test_path)],
                         capture_output=True, text=True)
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    run.check_returncode()
finally:
    DB.unlink(missing_ok=True)

...                                                                      [100%]
3 passed in 0.48s



## Things to try

Optional : change the bus or part identifier and run the example again.